# Cleaning and filtering

This adapted notebook implements a lightweight rule-based language filtering pipeline
for conversational tree-structured data (e.g., ShareGPT-style datasets).

Goal:
- Load raw conversation trees
- Detect English prompts with deterministic heuristics
- Filter non-English trees
- Save the cleaned dataset
- Inspect random sample prompts

Adjustments in this version:
- configuration, processing logic, diagnostics, and execution are separated cleanly
- the stopword heuristic remains the primary decision rule
- Lingua is optional and can run in `support_only` or `strict` mode
- extra diagnostics make heuristic/Lingua disagreements visible


In [75]:
# ----------------------------
# IMPORTS + CONFIG
# ----------------------------

import re
import json
import random
import unicodedata
from pathlib import Path
from time import perf_counter
from lingua import Language, LanguageDetectorBuilder


def find_project_root(start=None, marker="01_data"):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root containing {marker!r}")


PROJECT_ROOT = find_project_root()

CONFIG = {
    "data_dir_raw": PROJECT_ROOT / "01_data" / "01_raw" / "messy_dataset",
    "data_dir_processed": PROJECT_ROOT / "01_data" / "02_processed",
    "input_files": [
        "sg_90k_part1.json",
        "sg_90k_part2.json",
    ],
    "lingua_language_codes": [
        "ENGLISH",
        "GERMAN",
        "DUTCH",
        "FRENCH",
        "SPANISH",
        "PORTUGUESE",
        "ITALIAN",
        "POLISH",
        "FINNISH",
        "CROATIAN",
        "INDONESIAN",
        "CHINESE",
    ],
    "lingua_min_confidence": 0.75,
    "output_file": "sharegpt_english.json",
    "sample_size": 10,
    "sample_max_chars": 700,
    "random_seed": 42,
    "min_words": 7,
    "min_english_stopwords": 3,
    "min_margin": 3,
    "min_latin_ratio": 0.8,
    "max_non_ascii_ratio": 0.25,
    "max_digit_ratio": 0.15,
    "lingua_mode": "support_only",
}


In [76]:
# ----------------------------
# STOPWORDS + LINGUA SETUP
# ----------------------------

WORD_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ']+")

STOPWORDS = {
    "en": {
        "the","and","to","of","a","in","that","is","it","for","on","with","as","this",
        "be","are","you","your","can","will","not","or","if","we","from","by","at","an",
        "have","has","do","does","what","which","when","where","how","why","about","would",
        "could","should","there","their","they","them","these","those","one","more","use",
        "using","make","write","explain","please","help"
    },
    "de": {
        "der","die","das","und","ist","ich","du","sie","er","es","wir","nicht","ein","eine",
        "zu","mit","auf","für","von","wie","was","warum","wenn","dass","kann","sind","haben",
        "werden","auch","oder","aber","bitte","über","hallo"
    },
    "nl": {
        "de","het","een","en","van","ik","je","niet","dat","dit","die","op","voor","met",
        "als","zijn","hebben","worden","wat","waar","waarom","hoe"
    },
    "vi": {
        "và","là","của","có","cho","trong","một","không","tôi","bạn","hãy","này","để"
    },
    "es": {
        "el","la","los","las","y","de","que","en","un","una","es","por","para","con","como","no","se"
    },
    "fr": {
        "le","la","les","et","de","des","du","un","une","est","dans","pour","avec","que","qui"
    },
    "pt": {
        "o","a","os","as","e","de","que","em","um","uma","é","para","com","como","não"
    },
    "it": {
        "il","lo","la","gli","le","e","di","che","in","un","una","è","per","con"
    },
}

LINGUA_LANGUAGES = [getattr(Language, code) for code in CONFIG["lingua_language_codes"]]
lingua_detector = LanguageDetectorBuilder.from_languages(*LINGUA_LANGUAGES).build()


def detect_language_lingua(text):
    text = (text or "").strip()
    if not text:
        return {
            "language": None,
            "language_code": None,
            "confidence": 0.0,
            "top_confidences": [],
        }

    lang = lingua_detector.detect_language_of(text)
    confidences = lingua_detector.compute_language_confidence_values(text)
    top_confidences = [
        {
            "language": str(item.language),
            "language_code": getattr(item.language.iso_code_639_1, "name", None),
            "confidence": round(item.value, 4),
        }
        for item in confidences[:3]
    ]

    if lang is None:
        return {
            "language": None,
            "language_code": None,
            "confidence": 0.0,
            "top_confidences": top_confidences,
        }

    top_conf = top_confidences[0]["confidence"] if top_confidences else 0.0
    return {
        "language": str(lang),
        "language_code": getattr(lang.iso_code_639_1, "name", None),
        "confidence": top_conf,
        "top_confidences": top_confidences,
    }


In [77]:
# ----------------------------
# CORE LOGIC
# ----------------------------

def get_first_user_prompt(tree):
    conversations = tree.get("conversations", [])
    for message in conversations:
        if message.get("from") == "human":
            return message.get("value", "")
    return ""


def normalize_words(text):
    return [w.lower().strip("'") for w in WORD_RE.findall(text or "")]


def count_stopwords(words, stopwords_dict):
    return {
        lang: sum(w in stopwords for w in words)
        for lang, stopwords in stopwords_dict.items()
    }


def get_language_scores_from_prompt(prompt, stopwords_dict):
    words = normalize_words(prompt)
    stopword_scores = count_stopwords(words, stopwords_dict)
    english_score = stopword_scores["en"]
    strongest_other = max(v for k, v in stopword_scores.items() if k != "en")
    return {
        "words": words,
        "word_count": len(words),
        "stopword_scores": stopword_scores,
        "english_score": english_score,
        "strongest_other": strongest_other,
    }


def latin_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return 0.0

    latin = 0
    for ch in letters:
        try:
            if "LATIN" in unicodedata.name(ch):
                latin += 1
        except ValueError:
            pass
    return latin / len(letters)


def looks_noisy_or_mixed(text, config):
    words = normalize_words(text)
    if len(words) < config["min_words"]:
        return True

    ascii_words = sum(w.isascii() for w in words)
    non_ascii_words = len(words) - ascii_words
    digit_like = sum(any(ch.isdigit() for ch in w) for w in words)

    non_ascii_ratio = non_ascii_words / len(words)
    digit_ratio = digit_like / len(words)

    return (
        non_ascii_ratio > config["max_non_ascii_ratio"]
        or digit_ratio > config["max_digit_ratio"]
    )


def is_probably_english_lingua(text, min_confidence=0.75):
    result = detect_language_lingua(text)
    return (
        result["language_code"] == "EN"
        and result["confidence"] >= min_confidence
    )


def diagnose_prompt(prompt, config, stopwords_dict):
    noisy = looks_noisy_or_mixed(prompt, config)
    latin = latin_ratio(prompt)
    scores = get_language_scores_from_prompt(prompt, stopwords_dict)

    heuristic_core_ok = (
        scores["word_count"] >= config["min_words"]
        and scores["english_score"] >= config["min_english_stopwords"]
        and (scores["english_score"] - scores["strongest_other"]) >= config["min_margin"]
    )

    heuristic_ok = (
        not noisy
        and latin >= config["min_latin_ratio"]
        and heuristic_core_ok
    )

    lingua_result = detect_language_lingua(prompt)
    lingua_ok = (
        lingua_result["language_code"] == "EN"
        and lingua_result["confidence"] >= config["lingua_min_confidence"]
    )

    return {
        "prompt": prompt,
        "noisy": noisy,
        "latin_ratio": latin,
        "scores": scores,
        "heuristic_ok": heuristic_ok,
        "lingua_result": lingua_result,
        "lingua_ok": lingua_ok,
    }


def is_english_tree(tree, config, stopwords_dict):
    prompt = get_first_user_prompt(tree)

    if looks_noisy_or_mixed(prompt, config):
        return False

    if latin_ratio(prompt) < config["min_latin_ratio"]:
        return False

    scores = get_language_scores_from_prompt(prompt, stopwords_dict)
    if scores["word_count"] < config["min_words"]:
        return False

    english_score = scores["english_score"]
    strongest_other = scores["strongest_other"]

    heuristic_ok = (
        english_score >= config["min_english_stopwords"]
        and (english_score - strongest_other) >= config["min_margin"]
    )

    if not heuristic_ok:
        return False

    lingua_mode = config.get("lingua_mode", "off")

    if lingua_mode == "strict":
        return is_probably_english_lingua(
            prompt,
            min_confidence=config["lingua_min_confidence"]
        )

    return True


def filter_english_trees(data, config, stopwords_dict):
    return [
        tree for tree in data
        if is_english_tree(tree, config, stopwords_dict)
    ]

In [78]:
# ----------------------------
# IO HELPERS
# ----------------------------

def load_json(path):
    try:
        with Path(path).open("r", encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"File not found: {path}")
        raise


def load_all_json(paths):
    data = []
    for path in paths:
        data.extend(load_json(path))
    return data


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)


In [79]:
# ----------------------------
# DIAGNOSTICS
# ----------------------------

def show_random_questions(data, n=10, max_chars=700, seed=None):
    if seed is not None:
        random.seed(seed)

    sample_size = min(n, len(data))
    for i in random.sample(range(len(data)), sample_size):
        tree = data[i]
        prompt = get_first_user_prompt(tree)
        print("=" * 100)
        print(f"Index: {i}")
        print(prompt[:max_chars])


def show_borderline_cases(data, config, stopwords_dict, n=10, max_chars=250):
    borderline = []
    for i, tree in enumerate(data):
        prompt = get_first_user_prompt(tree)
        diag = diagnose_prompt(prompt, config, stopwords_dict)
        scores = diag["scores"]
        margin = scores["english_score"] - scores["strongest_other"]
        if diag["heuristic_ok"]:
            borderline.append({
                "index": i,
                "prompt": prompt,
                "word_count": scores["word_count"],
                "english_score": scores["english_score"],
                "strongest_other": scores["strongest_other"],
                "margin": margin,
                "lingua_code": diag["lingua_result"]["language_code"],
                "lingua_confidence": diag["lingua_result"]["confidence"],
            })

    print(f"Borderline cases: {len(borderline)}")

    borderline = sorted(
        borderline,
        key=lambda x: (x["margin"], x["english_score"], x["word_count"])
    )

    for case in borderline[:n]:
        print("=" * 100)
        print(f"Index: {case['index']}")
        print(f"Words: {case['word_count']}")
        print(f"EN score: {case['english_score']}")
        print(f"Strongest other: {case['strongest_other']}")
        print(f"Margin: {case['margin']}")
        print(f"Lingua: {case['lingua_code']} ({case['lingua_confidence']})")
        print(case["prompt"][:max_chars])


def show_lingua_disagreements(data, config, stopwords_dict, n=10, max_chars=250):
    disagreements = []
    for i, tree in enumerate(data):
        prompt = get_first_user_prompt(tree)
        diag = diagnose_prompt(prompt, config, stopwords_dict)
        if diag["heuristic_ok"] and not diag["lingua_ok"]:
            disagreements.append({
                "index": i,
                "prompt": prompt,
                "latin_ratio": diag["latin_ratio"],
                "english_score": diag["scores"]["english_score"],
                "strongest_other": diag["scores"]["strongest_other"],
                "lingua": diag["lingua_result"],
            })

    print(f"Lingua disagreements: {len(disagreements)}")

    for case in disagreements[:n]:
        print("=" * 100)
        print(f"Index: {case['index']}")
        print(f"Latin ratio: {case['latin_ratio']:.3f}")
        print(f"EN stopwords: {case['english_score']}")
        print(f"Strongest other: {case['strongest_other']}")
        print(case["lingua"])
        print(case["prompt"][:max_chars])


In [80]:
# ----------------------------
# MINI TESTS
# ----------------------------

example_en = {
    "conversations": [
        {"from": "human", "value": "How can I improve my Python code for data analysis?"},
        {"from": "assistant", "value": "You can start by..."}
    ]
}

example_de = {
    "conversations": [
        {"from": "human", "value": "Wie kann ich meinen Python Code verbessern?"},
        {"from": "assistant", "value": "Du kannst..."}
    ]
}

example_short = {
    "conversations": [
        {"from": "human", "value": "Hello there"},
    ]
}

assert get_first_user_prompt(example_en) == "How can I improve my Python code for data analysis?"
assert normalize_words("Hello, World!") == ["hello", "world"]

scores = count_stopwords(["how", "can", "i", "improve", "my", "code"], STOPWORDS)
assert scores["en"] >= 2

result_en = is_english_tree(example_en, CONFIG, STOPWORDS)
result_de = is_english_tree(example_de, CONFIG, STOPWORDS)
result_short = is_english_tree(example_short, CONFIG, STOPWORDS)

print("EN:", result_en)
print("DE:", result_de)
print("SHORT:", result_short)
print(diagnose_prompt(example_en["conversations"][0]["value"], CONFIG, STOPWORDS))
print(detect_language_lingua("How can I improve my Python code for data analysis?"))
print(detect_language_lingua("czy znasz narzędzie do monitorowania infrastruktury IT - Splunk?"))
print(detect_language_lingua("帮我写一个利用convlstm进行时间序列图像的目标检测"))

assert result_en is True
assert result_de is False
assert result_short is False

print("All mini-tests passed.")


EN: True
DE: False
SHORT: False
{'prompt': 'How can I improve my Python code for data analysis?', 'noisy': False, 'latin_ratio': 1.0, 'scores': {'words': ['how', 'can', 'i', 'improve', 'my', 'python', 'code', 'for', 'data', 'analysis'], 'word_count': 10, 'stopword_scores': {'en': 3, 'de': 0, 'nl': 0, 'vi': 0, 'es': 0, 'fr': 0, 'pt': 0, 'it': 0}, 'english_score': 3, 'strongest_other': 0}, 'heuristic_ok': True, 'lingua_result': {'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.5386, 'top_confidences': [{'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.5386}, {'language': 'Language.GERMAN', 'language_code': 'DE', 'confidence': 0.1118}, {'language': 'Language.DUTCH', 'language_code': 'NL', 'confidence': 0.0726}]}, 'lingua_ok': False}
{'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.5386, 'top_confidences': [{'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.5386}, {'language': 'Language.GERMAN', 'langua

In [81]:
# ----------------------------
# PATHS
# ----------------------------

input_paths = [
    CONFIG["data_dir_raw"] / filename
    for filename in CONFIG["input_files"]
]

output_path = CONFIG["data_dir_processed"] / CONFIG["output_file"]

print("Input paths:")
for path in input_paths:
    print("-", path)

print()
print("Output path:")
print("-", output_path)

Input paths:
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\01_raw\messy_dataset\sg_90k_part1.json
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\01_raw\messy_dataset\sg_90k_part2.json

Output path:
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\02_processed\sharegpt_english.json


In [82]:
# ----------------------------
# LOAD / SAVE DATA
# ----------------------------

start = perf_counter()

data = load_all_json(input_paths)
english_data = filter_english_trees(data, CONFIG, STOPWORDS)

duration = perf_counter() - start

print(f"Original: {len(data)}")
print(f"English only: {len(english_data)}")
print(f"Removed: {len(data) - len(english_data)}")
print(f"Time: {duration:.2f}s")
print(f"Trees/sec: {len(data)/duration:.0f}")


Original: 90665
English only: 50844
Removed: 39821
Time: 25.62s
Trees/sec: 3539


In [83]:
# ----------------------------
# RANDOM QUESTIONS
# ----------------------------

show_random_questions(
    english_data,
    n=CONFIG["sample_size"],
    max_chars=CONFIG["sample_max_chars"],
    seed=CONFIG["random_seed"],
)


Index: 41905
write a 5 paragraph text CEFR B1 level using the following words: 
on the whole
open
operate
organ
organization
outstanding
part-time
patient
peculiar
place
point
point 
position 
potential
preference
present
pressure
prime
principal
promote
put up with sth/sb 
reach
record 
reflect
register
regret
relate
relax 
remark 
remote
replace
rescue
respect
retire
right
role
room
rough 
rule
rule
run
rush
satisfy
scale
scene
schedule 
sensitive
service 
set
setting
shortly
shut (sth) down or shut down (sth)
similarity
slow (sb/sth) down / up or slow down/up (sb/sth) 
so-called
society
sooner or later 
spectacular
speech
spoil
start (sth) off or start off (sth)
style
sum up (sth/sb) or sum (sth/sb) u
Index: 7296
This code creates a new folder for every 15 minutes and stores the copied files in that folder. The folder name is generated based on the start time of the 15-minute interval. But In each folder, a CSV file and N number of images are copied. CSV file contains the N number o

In [84]:
# ----------------------------
# BORDERLINE CASES
# ----------------------------

show_borderline_cases(
    english_data,
    config=CONFIG,
    stopwords_dict=STOPWORDS,
    n=30,
    max_chars=300,
)


Borderline cases: 50844
Index: 136
Words: 7
EN score: 3
Strongest other: 0
Margin: 3
Lingua: EN (0.9756)
How should I study to rememeber everything
Index: 159
Words: 7
EN score: 3
Strongest other: 0
Margin: 3
Lingua: EN (0.7296)
tell me about the stages of enlightment
Index: 653
Words: 7
EN score: 3
Strongest other: 0
Margin: 3
Lingua: EN (0.2613)
Can I query chatgpt using an api? 
Index: 695
Words: 7
EN score: 3
Strongest other: 0
Margin: 3
Lingua: EN (0.5636)
please tell me more about apache tika
Index: 758
Words: 7
EN score: 3
Strongest other: 0
Margin: 3
Lingua: EN (0.844)
This is about Blog site Though Tide 
Index: 784
Words: 7
EN score: 3
Strongest other: 0
Margin: 3
Lingua: EN (0.5921)
why did musashi write the five rings?
Index: 1124
Words: 7
EN score: 3
Strongest other: 0
Margin: 3
Lingua: EN (0.8703)
2 / 2"what is SoP for study abroad students"
Index: 1464
Words: 7
EN score: 3
Strongest other: 0
Margin: 3
Lingua: EN (0.9621)
Development of physics from Electromagnetics to Rel

In [85]:
# ----------------------------
# HEURISTIC / LINGUA DISAGREEMENTS
# ----------------------------

show_lingua_disagreements(
    english_data,
    config=CONFIG,
    stopwords_dict=STOPWORDS,
    n=20,
    max_chars=300,
)


Lingua disagreements: 5159
Index: 8
Latin ratio: 1.000
EN stopwords: 5
Strongest other: 1
{'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.6852, 'top_confidences': [{'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.6852}, {'language': 'Language.DUTCH', 'language_code': 'NL', 'confidence': 0.1241}, {'language': 'Language.POLISH', 'language_code': 'PL', 'confidence': 0.0509}]}
how do I add multiple new columns in m for power query or power bi?
Index: 11
Latin ratio: 1.000
EN stopwords: 4
Strongest other: 1
{'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.4097, 'top_confidences': [{'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.4097}, {'language': 'Language.DUTCH', 'language_code': 'NL', 'confidence': 0.1693}, {'language': 'Language.ITALIAN', 'language_code': 'IT', 'confidence': 0.1061}]}
Java add to the arraylist of a class type
Index: 12
Latin ratio: 1.000
EN stopwords: 4
Strongest other: 0
{'lan

In [86]:
# ----------------------------
# OPTIONAL CONFIG COMPARISON
# ----------------------------

configs_to_compare = [
    {
        "name": "baseline",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 1,
        "min_latin_ratio": 0.0,
        "max_non_ascii_ratio": 1.0,
        "max_digit_ratio": 1.0,
        "use_lingua": False,
        "lingua_mode": "support_only",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
    {
        "name": "higher_margin",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 2,
        "min_latin_ratio": 0.0,
        "max_non_ascii_ratio": 1.0,
        "max_digit_ratio": 1.0,
        "use_lingua": False,
        "lingua_mode": "support_only",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
    {
        "name": "latin_ratio_check",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 1,
        "min_latin_ratio": 0.8,
        "max_non_ascii_ratio": 1.0,
        "max_digit_ratio": 1.0,
        "use_lingua": False,
        "lingua_mode": "support_only",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
    {
        "name": "strict_lingua",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 2,
        "min_latin_ratio": 0.8,
        "max_non_ascii_ratio": 0.25,
        "max_digit_ratio": 0.15,
        "use_lingua": True,
        "lingua_mode": "strict",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
    {
        "name": "support_only_lingua",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 2,
        "min_latin_ratio": 0.8,
        "max_non_ascii_ratio": 0.25,
        "max_digit_ratio": 0.15,
        "lingua_mode": "support_only",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
    {
        "name": "strict_lingua",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 2,
        "min_latin_ratio": 0.8,
        "max_non_ascii_ratio": 0.25,
        "max_digit_ratio": 0.15,
        "lingua_mode": "strict",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
]

for cfg in configs_to_compare:
    merged_cfg = {**CONFIG, **cfg}
    filtered = filter_english_trees(data, merged_cfg, STOPWORDS)
    kept_ratio = len(filtered) / len(data)
    print("=" * 80)
    print(merged_cfg["name"])
    print(f"English only: {len(filtered)}")
    print(f"Removed: {len(data) - len(filtered)}")
    print(f"Kept ratio: {kept_ratio:.3f}")


baseline
English only: 58921
Removed: 31744
Kept ratio: 0.650
higher_margin
English only: 57276
Removed: 33389
Kept ratio: 0.632
latin_ratio_check
English only: 57613
Removed: 33052
Kept ratio: 0.635
strict_lingua
English only: 47961
Removed: 42704
Kept ratio: 0.529
support_only_lingua
English only: 56002
Removed: 34663
Kept ratio: 0.618
strict_lingua
English only: 47961
Removed: 42704
Kept ratio: 0.529


## Decision on filtering

Both heuristic filtering and Lingua are used, but strict_lingua was selected because it required agreement between the two methods and therefore better removed mixed or noisy prompts. This trade-off favored cleaner, more structurally usable English data over a higher retention rate.

I set lingua_mode to "strict" so that I can continue working with this cleaner final dataset.

In [87]:
# ----------------------------
# FINAL FILTER CHOICE
# ----------------------------

FINAL_CONFIG = CONFIG.copy()
FINAL_CONFIG.update({
    "min_words": 5,
    "min_english_stopwords": 2,
    "min_margin": 2,
    "min_latin_ratio": 0.8,
    "max_non_ascii_ratio": 0.25,
    "max_digit_ratio": 0.15,
    "lingua_mode": "strict",
    "lingua_min_confidence": 0.75,
})

english_data = filter_english_trees(data, FINAL_CONFIG, STOPWORDS)

print(f"Final English-only dataset: {len(english_data)}")
print(f"Removed: {len(data) - len(english_data)}")
print(f"Kept ratio: {len(english_data) / len(data):.3f}")

Final English-only dataset: 47961
Removed: 42704
Kept ratio: 0.529


In [88]:
# ----------------------------
# SAVE
# ----------------------------

output_path = FINAL_CONFIG["data_dir_processed"] / FINAL_CONFIG["output_file"]
save_json(english_data, output_path)
